# Pipeline Kiểm Tra Mô Hình MoE + Q-Former GoPro Deblurring
Notebook này kiểm tra toàn bộ các khối chức năng:
1. Khởi tạo mô hình `MoEQFormerDeblurNet` và kiểm tra kích thước Tensor.
2. Kiểm tra nạp dữ liệu cặp ảnh (Blur / Sharp) từ thư mục `gopro-data/train`.
3. Chạy thử nghiệm 1 bước tính toán hàm mất mát (Charbonnier Loss + MoE Balance Loss) và kiểm tra cập nhật đạo hàm.

In [ ]:
import sys
import os
import torch
import matplotlib.pyplot as plt

# Thêm thư mục gốc vào path để import modules
sys.path.append(os.path.abspath('..'))

from models.moe_qformer_net import MoEQFormerDeblurNet
from utils.dataset import GoProDeblurDataset
from utils.metrics import CharbonnierLoss, calc_psnr, calc_ssim, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Thiết bị đang dùng: {device}")

In [ ]:
# 1. Test shape mô hình MoEQFormerDeblurNet
model = MoEQFormerDeblurNet(num_feats=64, num_queries=16, num_experts=3).to(device)
print(f"Tổng số tham số của mô hình: {count_parameters(model):,}")

# Giả lập 1 batch gồm 2 ảnh (Batch=2, Channels=3, H=128, W=128)
x_dummy = torch.randn(2, 3, 128, 128, device=device)
out_dummy, aux_loss = model(x_dummy)

print("Kích thước Input   :", x_dummy.shape)
print("Kích thước Output  :", out_dummy.shape)
print("MoE Balance Loss   :", aux_loss.item())
assert x_dummy.shape == out_dummy.shape, "Lỗi mismatch kích thước!"
print("--> Test forward mô hình THÀNH CÔNG!")

In [ ]:
# 2. Test nạp dữ liệu từ thư mục gopro-data thật
GOPRO_TRAIN_DIR = '../gopro-data/train'

if os.path.exists(GOPRO_TRAIN_DIR):
    dataset = GoProDeblurDataset(GOPRO_TRAIN_DIR, crop_size=256, is_train=True)
    print(f"Đã tìm thấy {len(dataset):,} ảnh trong tập GoPro Train.")
    blur_img, sharp_img = dataset[0]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(blur_img.permute(1, 2, 0).numpy())
    axes[0].set_title("Blur Image (Ảnh Mờ Đầu Vào)")
    axes[0].axis("off")

    axes[1].imshow(sharp_img.permute(1, 2, 0).numpy())
    axes[1].set_title("Sharp Image (Ảnh Gốc Nét)")
    axes[1].axis("off")
    plt.show()
else:
    print("Chưa tìm thấy thư mục gopro-data/train. Hãy kiểm tra lại thư mục!")

In [ ]:
# 3. Test 1 bước tính Loss và Backward
criterion = CharbonnierLoss()
loss_recon = criterion(out_dummy, x_dummy)
total_loss = loss_recon + 0.01 * aux_loss

total_loss.backward()
print(f"Reconstruction Loss: {loss_recon.item():.4f}")
print(f"Total Loss         : {total_loss.item():.4f}")
print("--> Test Backward và Gradient THÀNH CÔNG! Pipeline sẵn sàng.")